In [1]:
import numpy as np

1. scale = (x_max-x_min)/(q_max-qmin)
2. zp = round(q_min - (x_min/ scale))
3. clip zp if it exceeds range

In [55]:
# Quantization
def quantize_tensor(tensor_val, scale, zero_point):
    rounded = np.round((tensor_val / scale))
    shifted_t = rounded + zero_point
    clipped_t = np.clip(shifted_t, -128, 127)
    return clipped_t.astype(np.int8)

In [56]:
def dequantize_tensor(quan_tensor, scale, zero_point):
    dequant = (quan_tensor - zero_point) * scale
    return dequant.astype(np.float32)

In [57]:
# function to calculate scale and zero_point
def calculate_scale_zero_point(tensor,q_min = -128, q_max = 127):
    x_min = np.min(tensor)
    x_max = np.max(tensor)
    x_diff = x_max - x_min
    q_diff = q_max - q_min
    if x_diff == 0:
        scale = 1e-8
    else:
        scale = x_diff / q_diff
    zero_point = np.round(q_min - (x_min / scale))
    zero_point = (np.clip(zero_point, q_min, q_max)).astype(np.int8)
    return (scale, zero_point)

In [62]:
# Function to output in desired format
def display_tensor(tensor,tensor_name, scale, zero_point,q_tensor,dq_tensor):
    print("Tensor Name: ",tensor_name)
    print("Tensor: ",tensor)
    print("Tensor Max: ",np.max(tensor))
    print("Tensor Min: ",np.min(tensor))
    print("Scale: ",scale)
    print("ZeroPoint: ",zero_point)
    print("Quantized Tensor: ",q_tensor)
    print("Dequantized Tensor: ",dq_tensor)
    print("Mean Absolute error: ",np.mean(np.abs(tensor - dq_tensor)))

In [65]:
def scale_zero_point_find(tensor,t_name):
    scale, zero_point = calculate_scale_zero_point(tensor)
    quantized_tensor = quantize_tensor(tensor, scale, zero_point)
    dequantized_tensor = dequantize_tensor(quantized_tensor, scale, zero_point)
    display_tensor(tensor, t_name, scale, zero_point,quantized_tensor,dequantized_tensor)


In [66]:
# Tensor 1
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)
scale_zero_point_find(t1,'t1')

Tensor Name:  t1
Tensor:  [-1.5 -0.8  0.   0.9  2.3]
Tensor Max:  2.3
Tensor Min:  -1.5
Scale:  0.01490196
ZeroPoint:  -27
Quantized Tensor:  [-128  -81  -27   33  127]
Dequantized Tensor:  [-1.505098   -0.80470586  0.          0.8941176  -1.52      ]
Mean Absolute error:  0.7671372


In [68]:
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)
scale_zero_point_find(t2,'t2')

Tensor Name:  t2
Tensor:  [0.1 0.5 1.2 2.  3.5]
Tensor Max:  3.5
Tensor Min:  0.1
Scale:  0.013333334
ZeroPoint:  -128
Quantized Tensor:  [-120  -90  -38   22  127]
Dequantized Tensor:  [ 0.10666667  0.50666666  1.2        -1.4133334  -0.01333333]
Mean Absolute error:  1.388


In [69]:
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)
scale_zero_point_find(t3,'t3')

Tensor Name:  t3
Tensor:  [-3.  -2.1 -1.4 -0.6 -0.1]
Tensor Max:  -0.1
Tensor Min:  -3.0
Scale:  0.011372549
ZeroPoint:  127
Quantized Tensor:  [-128  -58    4   74  118]
Dequantized Tensor:  [ 0.01137255  0.807451   -1.3988236  -0.6027451  -0.10235295]
Mean Absolute error:  1.1850195


In [70]:
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)
scale_zero_point_find(t4,'t4')

Tensor Name:  t4
Tensor:  [5. 5. 5.]
Tensor Max:  5.0
Tensor Min:  5.0
Scale:  1e-08
ZeroPoint:  -128
Quantized Tensor:  [127 127 127]
Dequantized Tensor:  [-1.e-08 -1.e-08 -1.e-08]
Mean Absolute error:  5.0


In [71]:
t5 = t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)
scale_zero_point_find(t5,'t5')

Tensor Name:  t5
Tensor:  [ 1.e-09  2.e-09 -1.e-09]
Tensor Max:  2e-09
Tensor Min:  -1e-09
Scale:  1.17647055e-11
ZeroPoint:  -43
Quantized Tensor:  [  42  127 -128]
Dequantized Tensor:  [ 9.9999997e-10 -1.0117647e-09 -9.9999997e-10]
Mean Absolute error:  1.0039215e-09
